May 2025
BC205: Algorithms for Bioinformatics.
Exercise III: Implementing a simplified version of the FastA algorithm
Gionnas- Masselos Theodoros
Chairetaki Antonia

We created dictionaries with k-mers and search for shared k-mers between query sequence and multiple target sequences. After that, we scored the matches allowing small gaps and we grouped them into clusters based on how close they are diagonally. Scoring was done with a simple score of +1/0 for matches/mismatches, we kept the ones that cover enough of the query sequence. These matches are then chained into high-scoring regions, allowing small gaps. Finally, the algorithm outputs a list of target sequences that have a total chained score above a 50% threshold of the query's length, along with their respective scores.
From the output we can observe that there is one main matching sequence. This sequence corresponds to YMR021C (MAC1) gene which is located on chromosome XIII of Saccharomyces cerevisiae.

In [37]:
from Bio import SeqIO
import collections

# Reads a FASTA file and returns a list of sequence records (SeqRecord objects)
def load_fasta_sequences(filepath):
    return list(SeqIO.parse(filepath, "fasta"))

# Generates a dictionary of all k-mers in a sequence with their positions
def get_kmers(seq_str, k):
    kmers = collections.defaultdict(list)
    for i in range(len(seq_str) - k + 1):
        kmer = seq_str[i:i + k]
        kmers[kmer].append(i)
    return kmers

# Finds matching k-mers between query and target, organized by diagonals (t_pos - q_pos)
def build_diagonals(query_seq_str, target_seq_str, k):
    diagonals = collections.defaultdict(list)
    query_kmers = get_kmers(query_seq_str, k)
    
    for t_pos in range(len(target_seq_str) - k + 1):
        target_kmer = target_seq_str[t_pos:t_pos+k]
        if target_kmer in query_kmers:
            for q_pos in query_kmers[target_kmer]:
                diag = t_pos - q_pos
                diagonals[diag].append((q_pos, t_pos))
    return diagonals

# Clusters nearby k-mer matches (on the same diagonal) into contiguous regions
# This function will now return segments that have already passed the 'm' threshold
# and are joined within 'g' gaps.
def get_high_scoring_segments(matches_on_diagonal, k, m, g):
    if not matches_on_diagonal:
        return []

    matches_on_diagonal.sort(key=lambda x: x[0]) # Sort by query position

    segments = []
    
    current_segment_start_q = matches_on_diagonal[0][0]
    current_segment_start_t = matches_on_diagonal[0][1]
    current_segment_score = 1
    
    current_q_end_for_overlap = current_segment_start_q + k - 1
    current_t_end_for_overlap = current_segment_start_t + k - 1

    for i in range(1, len(matches_on_diagonal)):
        next_q_pos, next_t_pos = matches_on_diagonal[i]
        
        prev_q_match_start = matches_on_diagonal[i-1][0]
        prev_t_match_start = matches_on_diagonal[i-1][1]

        query_gap_to_prev_kmer = next_q_pos - (prev_q_match_start + k)
        target_gap_to_prev_kmer = next_t_pos - (prev_t_match_start + k)

        if query_gap_to_prev_kmer <= g and target_gap_to_prev_kmer <= g and \
           query_gap_to_prev_kmer >= -(k-1) and target_gap_to_prev_kmer >= -(k-1):
            current_segment_score += 1
            current_q_end_for_overlap = next_q_pos + k - 1
            current_t_end_for_overlap = next_t_pos + k - 1
        else:
            if current_segment_score >= m:
                segments.append({
                    'q_start': current_segment_start_q,
                    'q_end': current_q_end_for_overlap,
                    't_start': current_segment_start_t,
                    't_end': current_t_end_for_overlap,
                    'score': current_segment_score
                })
            
            current_segment_start_q = next_q_pos
            current_segment_start_t = next_t_pos
            current_segment_score = 1
            current_q_end_for_overlap = next_q_pos + k - 1
            current_t_end_for_overlap = next_t_pos + k - 1

    if current_segment_score >= m:
        segments.append({
            'q_start': current_segment_start_q,
            'q_end': current_q_end_for_overlap,
            't_start': current_segment_start_t,
            't_end': current_t_end_for_overlap,
            'score': current_segment_score
        })
    return segments

def generate_gapped_alignment(query_seq_str, target_seq_str, aligned_segments):

    gapped_query = []
    gapped_target = []

    last_q_end = -1 # Tracks the last aligned base position in query
    last_t_end = -1 # Tracks the last aligned base position in target

    # Handle leading unaligned part
    if aligned_segments:
        first_segment = aligned_segments[0]
        # Pad from beginning of sequence to start of first segment
        for _ in range(first_segment['q_start']):
            gapped_query.append('-')
            gapped_target.append('-')

    for i, segment in enumerate(aligned_segments):
        q_start, q_end = segment['q_start'], segment['q_end']
        t_start, t_end = segment['t_start'], segment['t_end']

        # Add gaps for unaligned regions between current segment and previous one
        # if this isn't the first segment
        if i > 0:
            # Gaps in query
            query_gap_length = q_start - (last_q_end + 1)
            for _ in range(query_gap_length):
                gapped_query.append('-')
            # Gaps in target
            target_gap_length = t_start - (last_t_end + 1)
            for _ in range(target_gap_length):
                gapped_target.append('-')
            
            # If one sequence has a larger gap than the other,
            # pad the shorter one to match the longer one's gap
            if query_gap_length > target_gap_length:
                for _ in range(query_gap_length - target_gap_length):
                    gapped_target.append('-')
            elif target_gap_length > query_gap_length:
                for _ in range(target_gap_length - query_gap_length):
                    gapped_query.append('-')

        # Add the aligned part of the current segment
        gapped_query.append(query_seq_str[q_start : q_end + 1])
        gapped_target.append(target_seq_str[t_start : t_end + 1])

        last_q_end = q_end
        last_t_end = t_end

    # Handle trailing unaligned part
    # We only pad if the query was fully captured in the alignment.
    # Otherwise, the length of the query_seq_str is the effective end.
    if last_q_end < len(query_seq_str) - 1:
        trailing_q_len = len(query_seq_str) - (last_q_end + 1)
        # Pad both query and target alignment with appropriate dashes for trailing unaligned parts
        for _ in range(trailing_q_len):
            gapped_query.append(query_seq_str[last_q_end + 1 + _])
            gapped_target.append('-') # Assume query is the reference for trailing gaps

    # Adjust overall alignment length to match the longest sequence.
    # If target extends beyond query's aligned part, add dashes to query.
    if last_t_end < len(target_seq_str) - 1:
        trailing_t_len = len(target_seq_str) - (last_t_end + 1)
        # Pad both query and target alignment with appropriate dashes for trailing unaligned parts
        for _ in range(trailing_t_len):
            gapped_target.append(target_seq_str[last_t_end + 1 + _])
            gapped_query.append('-') # Assume target is the reference for trailing gaps

    return "".join(gapped_query), "".join(gapped_target)


# Main simplified FastA algorithm
def fasta_simplified(query_record, target_records, k=5, m=20, g=3, min_total_score_percentage=0.50):
    results = []
    query_seq_str = str(query_record.seq)
    min_score_required = int(len(query_seq_str) * min_total_score_percentage)

    for target_record in target_records:
        target_id = target_record.id
        target_seq_str = str(target_record.seq)
        
        diagonals = build_diagonals(query_seq_str, target_seq_str, k)
        
        all_candidate_segments_for_target = []
        for diag, matches in diagonals.items():
            segments_on_diag = get_high_scoring_segments(matches, k, m, g)
            all_candidate_segments_for_target.extend(segments_on_diag)

        # Sort all segments by query start position to prepare for overall chaining
        all_candidate_segments_for_target.sort(key=lambda x: x['q_start'])

        final_aligned_regions_info = [] # Store dictionaries for easier access
        if not all_candidate_segments_for_target:
            continue

        # Initialize the first combined region
        current_region = {
            'q_start': all_candidate_segments_for_target[0]['q_start'],
            'q_end': all_candidate_segments_for_target[0]['q_end'],
            't_start': all_candidate_segments_for_target[0]['t_start'],
            't_end': all_candidate_segments_for_target[0]['t_end'],
            'score': all_candidate_segments_for_target[0]['score'],
            'segments': [all_candidate_segments_for_target[0]] # Store individual segments
        }

        # Chain segments across different diagonals if gaps are small enough
        for i in range(1, len(all_candidate_segments_for_target)):
            next_segment = all_candidate_segments_for_target[i]

            query_gap = next_segment['q_start'] - current_region['q_end'] - 1
            target_gap = next_segment['t_start'] - current_region['t_end'] - 1

            if query_gap <= g and target_gap <= g and \
               query_gap >= -(k-1) and target_gap >= -(k-1): # Allow overlaps
                
                # Update the region's overall span and total score
                current_region['q_end'] = max(current_region['q_end'], next_segment['q_end'])
                current_region['t_end'] = max(current_region['t_end'], next_segment['t_end'])
                current_region['score'] += next_segment['score']
                current_region['segments'].append(next_segment)
            else:
                final_aligned_regions_info.append(current_region)
                
                # Start a new region
                current_region = {
                    'q_start': next_segment['q_start'],
                    'q_end': next_segment['q_end'],
                    't_start': next_segment['t_start'],
                    't_end': next_segment['t_end'],
                    'score': next_segment['score'],
                    'segments': [next_segment]
                }
        
        final_aligned_regions_info.append(current_region) # Add the last region

        # Filter final regions by the overall score prerequisite and generate alignments
        for region_info in final_aligned_regions_info:
            if region_info['score'] >= min_score_required:
                # We need the segments sorted by query start for generate_gapped_alignment
                region_info['segments'].sort(key=lambda x: x['q_start'])
                
                gapped_query, gapped_target = generate_gapped_alignment(
                    query_seq_str, target_seq_str, region_info['segments']
                )
                
                results.append({
                    'target_id': target_id,
                    'total_score': region_info['score'],
                    'gapped_query': gapped_query,
                    'gapped_target': gapped_target
                })

    return results

# Running the FastA Search and Printing Alignments

if __name__ == "__main__":
    query_file = "query.fa"
    targets_file = "all_yeast_genes_minplus1k.fa"

    try:
        query_records = load_fasta_sequences(query_file)
        target_records = load_fasta_sequences(targets_file)

        if not query_records:
            raise ValueError("No query sequence found in query.fa")

        query_record = query_records[0]
        print(f"Loaded query sequence (ID: {query_record.id}, Length: {len(query_record.seq)})")
        print(f"Loaded {len(target_records)} target sequences.")

        matched_results = fasta_simplified(
            query_record,
            target_records,
            k=5,
            m=20,
            g=3,
            min_total_score_percentage=0.50
        )

        if matched_results:
            print("\n--- Matched Sequences (Total Score >= 50% of Query Length) ---")
            for i, result in enumerate(matched_results, 1):
                print(f"\nMatch {i}:")
                print(f"Target ID: {result['target_id']}")
                print(f"Total Score: {result['total_score']}")
                # Using f-strings to include query length for score percentage
                print(f"Score %: {result['total_score'] / len(query_record.seq):.2f}% ({result['total_score']}/{len(query_record.seq)})")
                print(f"Query : {result['gapped_query']}")
                print(f"Target: {result['gapped_target']}")
        else:
            print("\nNo sequences matched the criteria.")

    except FileNotFoundError:
        print(f"Error: Ensure '{query_file}' and '{targets_file}' are in the same directory.")
    except Exception as e:
        print(f"An error occurred: {e}")

Loaded query sequence (ID: QUERY, Length: 2017)
Loaded 5765 target sequences.

--- Matched Sequences (Total Score >= 50% of Query Length) ---

Match 1:
Target ID: YMR021C
Total Score: 1673
Score %: 0.83% (1673/2017)
Query : -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------GGAAGACCAAATGATCATTAACCTGCTACCTGCTTTCAGAATCCTAATGATTTCAAAATGGATAACTTTACCTGTCTCTCAAATTCAGGTCTATTATCTCTCCACAAGATGCAAGCATCAATGTTGGCACCACTTTCGATATTGGGCTCACTCAACATGCTCATAACACTTAATAGAATTTTTTCTACACTTTGCACTGGCGACCATCTTTCTTCCGCTAATTCGTACATGTTAGGATCATCACCAGGGGAGTGTAGAATGGATATGCACACTTCCCCATTTGGATAAATATTTGGATGTAGTATGCTGGGTGTGAAAGTAAGTTTAGGTGGAGATAACGGATAGTCTTTAGGAAACTCTAGCTTAGCATTAAAAACACCATCAGCGTATGGCGTATCTGGAGGCCCTTGAATTAGGCAGTCCCAAATGAATATGTTATTCTCCGATTTGGGACCAGCCACTATACCAGGTGGAGAATCTTTAATTAACTGTTGAAGCTCCTTGAGGAGACGTTTCTGAGCGGTTTTCGACATGCTATGCCCTTCCAAATTAC

Below, you can see the same code we only make minor changes to use only the most common k-mers instead of the full list, however it didn't make a huge difference as it runs only 2 seconds faster than the original code above and we still take exactly the same results. 

In [3]:
from Bio import SeqIO
import collections

# Reads a FASTA file and returns a list of sequence records (SeqRecord objects)
def load_fasta_sequences(filepath):
    return list(SeqIO.parse(filepath, "fasta"))

# Generates a dictionary of k-mers and their positions,
# filtering by a minimum occurrence count.
def get_kmers(seq_str, k, min_kmer_occurrence=1): 
    all_kmers_with_positions = collections.defaultdict(list)
    kmer_counts = collections.defaultdict(int)

    # count all k-mers and store all positions
    for i in range(len(seq_str) - k + 1):
        kmer = seq_str[i:i + k]
        kmer_counts[kmer] += 1
        all_kmers_with_positions[kmer].append(i)
    
    #filter for common k-mers based on min_kmer_occurrence
    common_kmers = {}
    for kmer, positions in all_kmers_with_positions.items():
        if kmer_counts[kmer] >= min_kmer_occurrence:
            common_kmers[kmer] = positions # Store positions only for common k-mers
            
    return common_kmers

# Finds matching k-mers between query and target, organized by diagonals (t_pos - q_pos)
def build_diagonals(query_seq_str, target_seq_str, k, query_min_kmer_occurrence=1):
    diagonals = collections.defaultdict(list)
    # Get only common k-mers from the query
    query_common_kmers = get_kmers(query_seq_str, k, query_min_kmer_occurrence)
    
    for t_pos in range(len(target_seq_str) - k + 1):
        target_kmer = target_seq_str[t_pos:t_pos+k]
        if target_kmer in query_common_kmers: # Check against common k-mers
            for q_pos in query_common_kmers[target_kmer]:
                diag = t_pos - q_pos
                diagonals[diag].append((q_pos, t_pos))
    return diagonals

# Clusters nearby k-mer matches (on the same diagonal) into regions
def get_high_scoring_segments(matches_on_diagonal, k, m, g):
    if not matches_on_diagonal:
        return []

    matches_on_diagonal.sort(key=lambda x: x[0]) # Sort by query position

    segments = []
    
    current_segment_start_q = matches_on_diagonal[0][0]
    current_segment_start_t = matches_on_diagonal[0][1]
    current_segment_score = 1
    
    current_q_end_for_overlap = current_segment_start_q + k - 1
    current_t_end_for_overlap = current_segment_start_t + k - 1

    for i in range(1, len(matches_on_diagonal)):
        next_q_pos, next_t_pos = matches_on_diagonal[i]
        
        prev_q_match_start = matches_on_diagonal[i-1][0]
        prev_t_match_start = matches_on_diagonal[i-1][1]

        query_gap_to_prev_kmer = next_q_pos - (prev_q_match_start + k)
        target_gap_to_prev_kmer = next_t_pos - (prev_t_match_start + k)

        if query_gap_to_prev_kmer <= g and target_gap_to_prev_kmer <= g and \
           query_gap_to_prev_kmer >= -(k-1) and target_gap_to_prev_kmer >= -(k-1):
            current_segment_score += 1
            current_q_end_for_overlap = next_q_pos + k - 1
            current_t_end_for_overlap = next_t_pos + k - 1
        else:
            if current_segment_score >= m:
                segments.append({
                    'q_start': current_segment_start_q,
                    'q_end': current_q_end_for_overlap,
                    't_start': current_segment_start_t,
                    't_end': current_t_end_for_overlap,
                    'score': current_segment_score
                })
            
            current_segment_start_q = next_q_pos
            current_segment_start_t = next_t_pos
            current_segment_score = 1
            current_q_end_for_overlap = next_q_pos + k - 1
            current_t_end_for_overlap = next_t_pos + k - 1

    if current_segment_score >= m:
        segments.append({
            'q_start': current_segment_start_q,
            'q_end': current_q_end_for_overlap,
            't_start': current_segment_start_t,
            't_end': current_t_end_for_overlap,
            'score': current_segment_score
        })
    return segments

# Main simplified FastA algorithm
def fasta_simplified(query_record, target_records, k=5, m=20, g=3, min_total_score_percentage=0.50, query_kmer_occurrence_threshold=1): 
    results_list_of_tuples = [] 
    
    query_seq_str = str(query_record.seq)
    min_score_required = int(len(query_seq_str) * min_total_score_percentage)

    for target_record in target_records:
        target_id = target_record.id
        target_seq_str = str(target_record.seq)
        
        # Pass the threshold to build_diagonals
        diagonals = build_diagonals(query_seq_str, target_seq_str, k, query_kmer_occurrence_threshold)
        
        all_candidate_segments_for_target = []
        for diag, matches in diagonals.items():
            segments_on_diag = get_high_scoring_segments(matches, k, m, g)
            all_candidate_segments_for_target.extend(segments_on_diag)

        all_candidate_segments_for_target.sort(key=lambda x: x['q_start'])

        current_region_q_start = -1
        current_region_q_end = -1
        current_region_t_start = -1
        current_region_t_end = -1
        current_region_score = 0
        
        has_current_region = False

        if not all_candidate_segments_for_target:
            continue

        first_segment = all_candidate_segments_for_target[0]
        current_region_q_start = first_segment['q_start']
        current_region_q_end = first_segment['q_end']
        current_region_t_start = first_segment['t_start']
        current_region_t_end = first_segment['t_end']
        current_region_score = first_segment['score']
        has_current_region = True

        for i in range(1, len(all_candidate_segments_for_target)):
            next_segment = all_candidate_segments_for_target[i]

            query_gap = next_segment['q_start'] - current_region_q_end - 1
            target_gap = next_segment['t_start'] - current_region_t_end - 1

            if query_gap <= g and target_gap <= g and \
               query_gap >= -(k-1) and target_gap >= -(k-1):
                
                current_region_q_end = max(current_region_q_end, next_segment['q_end'])
                current_region_t_end = max(current_region_t_end, next_segment['t_end'])
                current_region_score += next_segment['score']
            else:
                if current_region_score >= min_score_required:
                    results_list_of_tuples.append((target_id, current_region_score))
                
                current_region_q_start = next_segment['q_start']
                current_region_q_end = next_segment['q_end']
                current_region_t_start = next_segment['t_start']
                current_region_t_end = next_segment['t_end']
                current_region_score = next_segment['score']
        
        if has_current_region and current_region_score >= min_score_required:
            results_list_of_tuples.append((target_id, current_region_score))

    return results_list_of_tuples


# Running the FastA Search with Common K-mers

if __name__ == "__main__":
    query_file = "query.fa"
    targets_file = "all_yeast_genes_minplus1k.fa"

    KMER_OCCURRENCE_THRESHOLD = 2 

    try:
        query_records = load_fasta_sequences(query_file)
        target_records = load_fasta_sequences(targets_file)

        if not query_records:
            raise ValueError("No query sequence found in query.fa")

        query_record = query_records[0]
        print(f"Loaded query sequence (ID: {query_record.id}, Length: {len(query_record.seq)})")
        print(f"Loaded {len(target_records)} target sequences.")

        matched_results = fasta_simplified(
            query_record,
            target_records,
            k=5,
            m=20,
            g=3,
            min_total_score_percentage=0.50,
            query_kmer_occurrence_threshold=KMER_OCCURRENCE_THRESHOLD 
        )

        if matched_results:
            print(f"\n--- Final Matched Sequences and Total Scores (using k-mers appearing >= {KMER_OCCURRENCE_THRESHOLD} times in query) ---")
            for target_id, score in matched_results:
                print(f"Target ID: {target_id}, Total Score: {score}")
        else:
            print("\nNo sequences matched the criteria.")

    except FileNotFoundError:
        print(f"Error: Ensure '{query_file}' and '{targets_file}' are in the same directory.")
    except Exception as e:
        print(f"An error occurred: {e}")

Loaded query sequence (ID: QUERY, Length: 2017)
Loaded 5765 target sequences.

--- Final Matched Sequences and Total Scores (using k-mers appearing >= 2 times in query) ---
Target ID: YMR021C, Total Score: 1459


Here we run the same code as before but with increased k, we tried various k values and observed that even if we increase the k approximatly 20 times, it still work pretty good, we have matches, but it doesn't make a huge difference in terms of time. Below you can observe an example with k=40, which is run in approximately 40 seconds almost the same with the previus code. 

In [1]:
from Bio import SeqIO
import collections

# Reads a FASTA file and returns a list of sequence records (SeqRecord objects)
def load_fasta_sequences(filepath):
    return list(SeqIO.parse(filepath, "fasta"))

# Generates a dictionary of k-mers and their positions,
# filtering by a minimum occurrence count.
def get_kmers(seq_str, k, min_kmer_occurrence=1): 
    all_kmers_with_positions = collections.defaultdict(list)
    kmer_counts = collections.defaultdict(int)

    # count all k-mers and store all positions
    for i in range(len(seq_str) - k + 1):
        kmer = seq_str[i:i + k]
        kmer_counts[kmer] += 1
        all_kmers_with_positions[kmer].append(i)
    
    #filter for common k-mers based on min_kmer_occurrence
    common_kmers = {}
    for kmer, positions in all_kmers_with_positions.items():
        if kmer_counts[kmer] >= min_kmer_occurrence:
            common_kmers[kmer] = positions # Store positions only for common k-mers
            
    return common_kmers

# Finds matching k-mers between query and target, organized by diagonals (t_pos - q_pos)
def build_diagonals(query_seq_str, target_seq_str, k, query_min_kmer_occurrence=1):
    diagonals = collections.defaultdict(list)
    # Get only common k-mers from the query
    query_common_kmers = get_kmers(query_seq_str, k, query_min_kmer_occurrence)
    
    for t_pos in range(len(target_seq_str) - k + 1):
        target_kmer = target_seq_str[t_pos:t_pos+k]
        if target_kmer in query_common_kmers: # Check against common k-mers
            for q_pos in query_common_kmers[target_kmer]:
                diag = t_pos - q_pos
                diagonals[diag].append((q_pos, t_pos))
    return diagonals

# Clusters nearby k-mer matches (on the same diagonal) into regions
def get_high_scoring_segments(matches_on_diagonal, k, m, g):
    if not matches_on_diagonal:
        return []

    matches_on_diagonal.sort(key=lambda x: x[0]) # Sort by query position

    segments = []
    
    current_segment_start_q = matches_on_diagonal[0][0]
    current_segment_start_t = matches_on_diagonal[0][1]
    current_segment_score = 1
    
    current_q_end_for_overlap = current_segment_start_q + k - 1
    current_t_end_for_overlap = current_segment_start_t + k - 1

    for i in range(1, len(matches_on_diagonal)):
        next_q_pos, next_t_pos = matches_on_diagonal[i]
        
        prev_q_match_start = matches_on_diagonal[i-1][0]
        prev_t_match_start = matches_on_diagonal[i-1][1]

        query_gap_to_prev_kmer = next_q_pos - (prev_q_match_start + k)
        target_gap_to_prev_kmer = next_t_pos - (prev_t_match_start + k)

        if query_gap_to_prev_kmer <= g and target_gap_to_prev_kmer <= g and \
           query_gap_to_prev_kmer >= -(k-1) and target_gap_to_prev_kmer >= -(k-1):
            current_segment_score += 1
            current_q_end_for_overlap = next_q_pos + k - 1
            current_t_end_for_overlap = next_t_pos + k - 1
        else:
            if current_segment_score >= m:
                segments.append({
                    'q_start': current_segment_start_q,
                    'q_end': current_q_end_for_overlap,
                    't_start': current_segment_start_t,
                    't_end': current_t_end_for_overlap,
                    'score': current_segment_score
                })
            
            current_segment_start_q = next_q_pos
            current_segment_start_t = next_t_pos
            current_segment_score = 1
            current_q_end_for_overlap = next_q_pos + k - 1
            current_t_end_for_overlap = next_t_pos + k - 1

    if current_segment_score >= m:
        segments.append({
            'q_start': current_segment_start_q,
            'q_end': current_q_end_for_overlap,
            't_start': current_segment_start_t,
            't_end': current_t_end_for_overlap,
            'score': current_segment_score
        })
    return segments

# Main simplified FastA algorithm
def fasta_simplified(query_record, target_records, k=40, m=20, g=3, min_total_score_percentage=0.50, query_kmer_occurrence_threshold=1): 
    results_list_of_tuples = [] 
    
    query_seq_str = str(query_record.seq)
    min_score_required = int(len(query_seq_str) * min_total_score_percentage)

    for target_record in target_records:
        target_id = target_record.id
        target_seq_str = str(target_record.seq)
        
        # Pass the threshold to build_diagonals
        diagonals = build_diagonals(query_seq_str, target_seq_str, k, query_kmer_occurrence_threshold)
        
        all_candidate_segments_for_target = []
        for diag, matches in diagonals.items():
            segments_on_diag = get_high_scoring_segments(matches, k, m, g)
            all_candidate_segments_for_target.extend(segments_on_diag)

        all_candidate_segments_for_target.sort(key=lambda x: x['q_start'])

        current_region_q_start = -1
        current_region_q_end = -1
        current_region_t_start = -1
        current_region_t_end = -1
        current_region_score = 0
        
        has_current_region = False

        if not all_candidate_segments_for_target:
            continue

        first_segment = all_candidate_segments_for_target[0]
        current_region_q_start = first_segment['q_start']
        current_region_q_end = first_segment['q_end']
        current_region_t_start = first_segment['t_start']
        current_region_t_end = first_segment['t_end']
        current_region_score = first_segment['score']
        has_current_region = True

        for i in range(1, len(all_candidate_segments_for_target)):
            next_segment = all_candidate_segments_for_target[i]

            query_gap = next_segment['q_start'] - current_region_q_end - 1
            target_gap = next_segment['t_start'] - current_region_t_end - 1

            if query_gap <= g and target_gap <= g and \
               query_gap >= -(k-1) and target_gap >= -(k-1):
                
                current_region_q_end = max(current_region_q_end, next_segment['q_end'])
                current_region_t_end = max(current_region_t_end, next_segment['t_end'])
                current_region_score += next_segment['score']
            else:
                if current_region_score >= min_score_required:
                    results_list_of_tuples.append((target_id, current_region_score))
                
                current_region_q_start = next_segment['q_start']
                current_region_q_end = next_segment['q_end']
                current_region_t_start = next_segment['t_start']
                current_region_t_end = next_segment['t_end']
                current_region_score = next_segment['score']
        
        if has_current_region and current_region_score >= min_score_required:
            results_list_of_tuples.append((target_id, current_region_score))

    return results_list_of_tuples


# Running the FastA Search with Common K-mers

if __name__ == "__main__":
    query_file = "query.fa"
    targets_file = "all_yeast_genes_minplus1k.fa"

    KMER_OCCURRENCE_THRESHOLD = 2 

    try:
        query_records = load_fasta_sequences(query_file)
        target_records = load_fasta_sequences(targets_file)

        if not query_records:
            raise ValueError("No query sequence found in query.fa")

        query_record = query_records[0]
        print(f"Loaded query sequence (ID: {query_record.id}, Length: {len(query_record.seq)})")
        print(f"Loaded {len(target_records)} target sequences.")

        matched_results = fasta_simplified(
            query_record,
            target_records,
            k=5,
            m=20,
            g=3,
            min_total_score_percentage=0.50,
            query_kmer_occurrence_threshold=KMER_OCCURRENCE_THRESHOLD 
        )

        if matched_results:
            print(f"\n--- Final Matched Sequences and Total Scores (using k-mers appearing >= {KMER_OCCURRENCE_THRESHOLD} times in query) ---")
            for target_id, score in matched_results:
                print(f"Target ID: {target_id}, Total Score: {score}")
        else:
            print("\nNo sequences matched the criteria.")

    except FileNotFoundError:
        print(f"Error: Ensure '{query_file}' and '{targets_file}' are in the same directory.")
    except Exception as e:
        print(f"An error occurred: {e}")

Loaded query sequence (ID: QUERY, Length: 2017)
Loaded 5765 target sequences.

--- Final Matched Sequences and Total Scores (using k-mers appearing >= 2 times in query) ---
Target ID: YMR021C, Total Score: 1459
